# Functional Enrichment

This optional notebook runs GO Biological Process enrichment for the module folders produced in notebook `modules`.

Set `ctx.run_enrichment = True` in the setup cell before running the enrichment step.

## Inputs




- RNA and protein Leiden module folders produced in `modules`


- Optional GO Biological Process gene sets when enrichment is enabled




## Outputs




- GO enrichment result folders for the RNA and protein module collections


- Cross-modal GO-overlap summary tables for downstream interpretation




## Pipeline




1. Load the RNA and protein Leiden module folders from `modules`.


2. Optionally run GO enrichment for each retained module folder.


3. Compare enriched GO terms across RNA and protein modules and save the overlap summaries.




## Setup




The first code cell extends the notebook import path to shared context helpers in `Pipeline/4_Network_Analysis/utils/network_and_gcc.py` and local functional-analysis helpers in `Pipeline/5_Functional_Analysis/utils/functional_enrichment.py`, then initializes the shared `AnalysisContext`.

In [ ]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
NETWORK_ANALYSIS_DIR = NOTEBOOK_DIR.parent / "4_Network_Analysis"
FUNCTIONAL_UTILS_DIR = NOTEBOOK_DIR / "utils"
for search_path in [FUNCTIONAL_UTILS_DIR, NETWORK_ANALYSIS_DIR]:
    if str(search_path) not in sys.path:
        sys.path.insert(0, str(search_path))

import utils.network_and_gcc as notebook_utils
import functional_enrichment as enrichment_utils

notebook_utils = importlib.reload(notebook_utils)
enrichment_utils = importlib.reload(enrichment_utils)

from utils.network_and_gcc import build_analysis_context
from functional_enrichment import go_enrichment_for_module_folder

ctx = build_analysis_context(notebook_name='functional_enrichment', run_cytoscape=False, run_enrichment=True, save_figures=True)


## Step 1: Run GO enrichment per module folder

This step calls `go_enrichment_for_module_folder` for the RNA and protein Leiden module folders and writes the GO outputs used in later ranking summaries.

In [ ]:
MIN_MODULE_GENES_FOR_ENRICHMENT = 20

rna_module_dir = ctx.notebook_output_path('modules', 'RNA_Leiden_Modules_1.0')
protein_module_dir = ctx.notebook_output_path('modules', 'Protein_Leiden_Modules_1.0')

rna_module_go = go_enrichment_for_module_folder(
    rna_module_dir,
    gene_sets='GO_Biological_Process_2023',
    output_dir=ctx.output_path('RNA_Leiden_Modules_1.0', 'GO'),
    pval_cutoff=0.05,
    padj_cutoff=0.05,
    min_genes=MIN_MODULE_GENES_FOR_ENRICHMENT,
    enabled=ctx.run_enrichment,
    )

protein_module_go = go_enrichment_for_module_folder(
    protein_module_dir,
    gene_sets='GO_Biological_Process_2023',
    output_dir=ctx.output_path('Protein_Leiden_Modules_1.0', 'GO'),
    pval_cutoff=0.05,
    padj_cutoff=0.05,
    min_genes=MIN_MODULE_GENES_FOR_ENRICHMENT,
    enabled=ctx.run_enrichment,
    )

print(f"RNA module GO sets: {len(rna_module_go)}")
print(f"Protein module GO sets: {len(protein_module_go)}")

Skipping module_8: 4 genes < min_genes=20
Wrote GO results for 7 modules to C:\Users\tiril\Master-Tiril\results\notebooks\04_functional_enrichment\RNA_Leiden_Modules_1.0\GO
Wrote GO results for 6 modules to C:\Users\tiril\Master-Tiril\results\notebooks\04_functional_enrichment\Protein_Leiden_Modules_1.0\GO
RNA module GO sets: 7
Protein module GO sets: 6


## Step 2: Compare RNA and protein GO-term overlap

This step compares enriched GO terms across all RNA and protein modules. For each RNA module it reports the protein module with the highest GO-term overlap and saves the pairwise overlap table for downstream use.

In [6]:
best_go_overlap_df, go_overlap_pairwise_df = enrichment_utils.summarize_cross_network_go_overlap(
    rna_module_go,
    prot_module_go,
    )

best_go_overlap_path = ctx.output_path('rna_to_protein_best_go_overlap.csv')
go_overlap_pairwise_path = ctx.output_path('rna_protein_go_overlap_pairwise.csv')

best_go_overlap_df.to_csv(best_go_overlap_path, index=False)
go_overlap_pairwise_df.to_csv(go_overlap_pairwise_path, index=False)

display(best_go_overlap_df)
print(f'Saved best-match GO overlap list to {best_go_overlap_path}')
print(f'Saved pairwise GO overlap table to {go_overlap_pairwise_path}')

if go_overlap_pairwise_df.empty:
    print('No GO overlap available to compare across RNA and protein modules.')

,RNA_Module,Best_Protein_Module,RNA_GO_Term_Count,Protein_GO_Term_Count,Shared_GO_Term_Count,Jaccard_Overlap,Shared_GO_Terms
0,module_1,module_5,46,47,7,0.081395,COPII-coated Vesicle Budding (GO:0090114); Cot...
1,module_2,module_5,21,47,8,0.133333,ERAD Pathway (GO:0036503); Endomembrane System...
2,module_3,module_1,0,37,0,0.000000,
3,module_4,module_1,7,37,0,0.000000,
4,module_5,module_4,10,20,2,0.071429,Extracellular Matrix Organization (GO:0030198)...
5,module_6,module_1,0,37,0,0.000000,
6,module_7,module_4,25,20,2,0.046512,Extracellular Matrix Organization (GO:0030198)...


Saved best-match GO overlap list to C:\Users\tiril\Master-Tiril\results\notebooks\04_functional_enrichment\rna_to_protein_best_go_overlap.csv
Saved pairwise GO overlap table to C:\Users\tiril\Master-Tiril\results\notebooks\04_functional_enrichment\rna_protein_go_overlap_pairwise.csv
